# Step 3 — Week 2: Chunking Strategies and Retrieval Quality

## Goals
- Implement multiple chunking strategies:
  - Fixed-size chunking
  - Semantic chunking
  - Overlap tuning
- Evaluate retrieval quality impacts with repeatable metrics

## Notes
This notebook uses a small in-memory corpus and `TF-IDF + cosine similarity` so you can run it quickly.
You can later swap in embedding models and vector DBs (e.g., Chroma) using the same evaluation loop.

In [20]:
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

### What this code does
This block imports the libraries used across the notebook.
- **Data + math tools**: `pandas`, `numpy`
- **Chunking tools**: LangChain text splitters (`Character`, `Recursive`, `Token`)
- **Retrieval tools**: TF-IDF and cosine similarity from scikit-learn

It also sets a pandas display option so table text is easier to read.

In [21]:
# Small synthetic corpus with distinct topical sections
DOCS = {
    "doc_rag": """
Retrieval-Augmented Generation (RAG) combines retrieval and generation. 
A retriever fetches relevant context chunks from a knowledge base. 
The generator then uses those chunks to answer with better factual grounding. 
Chunk quality strongly affects retrieval hit rate and downstream answer quality.
""",
    "doc_chunking": """
Chunking splits documents into smaller pieces for indexing. 
Fixed-size chunking is simple and fast but may split ideas across boundaries. 
Overlap preserves context continuity between adjacent chunks. 
Semantic chunking tries to split on meaning shifts, often near sentence boundaries.
""",
    "doc_eval": """
Evaluation should measure retrieval and answer quality. 
Common retrieval metrics include hit rate at k and mean reciprocal rank. 
Faithfulness checks whether answers are grounded in retrieved evidence. 
Format compliance measures whether outputs follow required structure.
""",
    "doc_vectors": """
Vector databases store embeddings for fast similarity search. 
Cosine similarity is commonly used to compare dense vectors. 
Metadata filters can narrow search to subsets of documents. 
Reranking can improve relevance after initial retrieval.
""",
}

QUERIES = [
    {"query": "What does RAG combine?", "relevant_doc": "doc_rag"},
    {"query": "Why does overlap help chunking?", "relevant_doc": "doc_chunking"},
    {"query": "Name a retrieval metric at k.", "relevant_doc": "doc_eval"},
    {"query": "What does cosine similarity compare?", "relevant_doc": "doc_vectors"},
    {"query": "What checks grounding in retrieved evidence?", "relevant_doc": "doc_eval"},
    {"query": "What can improve relevance after initial retrieval?", "relevant_doc": "doc_vectors"},
]

print(f"Loaded {len(DOCS)} docs and {len(QUERIES)} evaluation queries.")

Loaded 4 docs and 6 evaluation queries.


### Concept: Evaluation setup
This block creates a small synthetic corpus and a query set with known relevant documents.
- Each query has a `relevant_doc` label (ground truth).
- This makes retrieval evaluation repeatable and comparable across chunking strategies.

Think of this as a controlled benchmark dataset for your chunking experiments.

## 1) Implement Chunking Strategies

In [22]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str


def get_langchain_splitter(strategy: str, overlap_words: int = 0):
    if strategy == "fixed":
        chunk_words = 30
        if overlap_words >= chunk_words:
            raise ValueError("overlap_words must be < 30 for fixed strategy")
        return CharacterTextSplitter(
            separator=" ",
            chunk_size=chunk_words,
            chunk_overlap=overlap_words,
            length_function=lambda s: len(s.split()),
        )

    if strategy == "semantic":
        return RecursiveCharacterTextSplitter(
            chunk_size=45,
            chunk_overlap=5,
            length_function=lambda s: len(s.split()),
            separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        )

    if strategy == "token":
        chunk_tokens = 40
        if overlap_words >= chunk_tokens:
            raise ValueError("overlap_words must be < 40 for token strategy")
        return TokenTextSplitter(
            chunk_size=chunk_tokens,
            chunk_overlap=overlap_words,
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_chunks(strategy: str, overlap_words: int = 0) -> List[Chunk]:
    chunks: List[Chunk] = []

    for doc_id, text in DOCS.items():
        splitter = get_langchain_splitter(strategy=strategy, overlap_words=overlap_words)
        parts = [chunk.strip() for chunk in splitter.split_text(text) if chunk.strip()]

        for idx, part in enumerate(parts):
            chunks.append(Chunk(chunk_id=f"{doc_id}_{idx}", doc_id=doc_id, text=part))

    return chunks


# Quick sanity preview
fixed_preview = build_chunks("fixed", overlap_words=10)
semantic_preview = build_chunks("semantic")
token_preview = build_chunks("token", overlap_words=5)
print(
    f"Fixed chunks: {len(fixed_preview)} | Semantic chunks: {len(semantic_preview)} | Token chunks: {len(token_preview)}"
 )
print("Sample fixed chunk:", fixed_preview[0].text[:120], "...")
print("Sample semantic chunk:", semantic_preview[0].text[:120], "...")
print("Sample token chunk:", token_preview[0].text[:120], "...")

Fixed chunks: 8 | Semantic chunks: 4 | Token chunks: 8
Sample fixed chunk: Retrieval-Augmented Generation (RAG) combines retrieval and generation. 
A retriever fetches relevant context chunks fro ...
Sample semantic chunk: Retrieval-Augmented Generation (RAG) combines retrieval and generation. 
A retriever fetches relevant context chunks fro ...
Sample token chunk: Retrieval-Augmented Generation (RAG) combines retrieval and generation. 
A retriever fetches relevant context chunks fro ...


### Concept: Building chunks with LangChain
This block defines how documents are split into chunks for each strategy.
- **Fixed**: word-like size windows with controllable overlap
- **Semantic-ish**: recursive splitting with sentence-aware separators
- **Token**: token-count-based splitting

`build_chunks(...)` applies the selected splitter to every document and stores metadata (`chunk_id`, `doc_id`, `text`).

## 2) Retrieval and Metrics

In [23]:
def retrieve_top_k(query: str, chunks: List[Chunk], k: int = 3) -> List[Tuple[Chunk, float]]:
    texts = [c.text for c in chunks]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    chunk_mat = vectorizer.fit_transform(texts)
    query_vec = vectorizer.transform([query])

    sims = cosine_similarity(query_vec, chunk_mat).flatten()
    top_idx = np.argsort(-sims)[:k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


def reciprocal_rank(retrieved_docs: List[str], gold_doc: str) -> float:
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if doc_id == gold_doc:
            return 1.0 / rank
    return 0.0


def hit_at_k(retrieved_docs: List[str], gold_doc: str, k: int) -> int:
    return int(gold_doc in retrieved_docs[:k])


def evaluate_retrieval(chunks: List[Chunk], queries: List[Dict[str, str]], k: int = 3) -> Dict[str, float]:
    hits = []
    mrrs = []

    for item in queries:
        ranked = retrieve_top_k(item["query"], chunks, k=k)
        ranked_doc_ids = [c.doc_id for c, _ in ranked]
        hits.append(hit_at_k(ranked_doc_ids, item["relevant_doc"], k))
        mrrs.append(reciprocal_rank(ranked_doc_ids, item["relevant_doc"]))

    return {
        "hit_rate@k": float(np.mean(hits)),
        "mrr": float(np.mean(mrrs)),
    }

### Concept: Retrieval scoring and metrics
This block implements the retrieval pipeline and evaluation metrics.
1. Convert chunk text into TF-IDF vectors.
2. Score query-to-chunk relevance with cosine similarity.
3. Take top-k chunks and map them to document IDs.
4. Compute quality metrics: **Hit Rate@k** and **MRR**.

These metrics let you compare chunking strategies objectively.

## 3) Overlap Tuning + Strategy Comparison

In [24]:
rows = []

# Fixed-size with overlap sweep
for ov in [0, 5, 10, 15]:
    ch = build_chunks("fixed", overlap_words=ov)
    metrics = evaluate_retrieval(ch, QUERIES, k=3)
    rows.append({
        "strategy": "fixed",
        "overlap_words": ov,
        "num_chunks": len(ch),
        **metrics,
    })

# Token-based with overlap sweep
for ov in [0, 5, 10, 15]:
    ch_tok = build_chunks("token", overlap_words=ov)
    metrics_tok = evaluate_retrieval(ch_tok, QUERIES, k=3)
    rows.append({
        "strategy": "token",
        "overlap_words": ov,
        "num_chunks": len(ch_tok),
        **metrics_tok,
    })

# Semantic baseline
ch_sem = build_chunks("semantic")
metrics_sem = evaluate_retrieval(ch_sem, QUERIES, k=3)
rows.append({
    "strategy": "semantic",
    "overlap_words": None,
    "num_chunks": len(ch_sem),
    **metrics_sem,
})

results_df = pd.DataFrame(rows).sort_values(["strategy", "overlap_words"], na_position="last")
results_df

,strategy,overlap_words,num_chunks,hit_rate@k,mrr
0,fixed,0.0,8,1.0,0.916667
1,fixed,5.0,8,1.0,0.916667
2,fixed,10.0,8,1.0,1.000000
3,fixed,15.0,8,1.0,1.000000
8,semantic,NaN,4,1.0,0.916667
4,token,0.0,8,1.0,0.888889
5,token,5.0,8,1.0,0.916667
6,token,10.0,8,1.0,0.916667
7,token,15.0,8,1.0,0.916667


### Concept: Strategy comparison experiment
This block runs the full experiment across configurations.
- Sweeps overlap values for **fixed** and **token** chunking
- Runs a baseline for **semantic** chunking
- Collects metrics and chunk counts into one results table

The resulting dataframe is your side-by-side comparison view.

In [25]:
best_row = results_df.sort_values(["hit_rate@k", "mrr"], ascending=False).iloc[0]
print("Best config:")
print(best_row)

print("\nInterpretation guide:")
print("- Higher hit_rate@k means relevant docs appear in top-k more often.")
print("- Higher MRR means relevant docs appear earlier in ranking.")
print("- More overlap usually increases chunk count and can improve recall at cost of index size.")

Best config:
strategy         fixed
overlap_words     10.0
num_chunks           8
hit_rate@k         1.0
mrr                1.0
Name: 2, dtype: object

Interpretation guide:
- Higher hit_rate@k means relevant docs appear in top-k more often.
- Higher MRR means relevant docs appear earlier in ranking.
- More overlap usually increases chunk count and can improve recall at cost of index size.


### Concept: Interpreting the winner
This block picks the best-performing configuration from the table.
- Sorts by `hit_rate@k` and `mrr` (higher is better)
- Prints the top row as the current best setup
- Provides quick interpretation guidance for the metrics

Use this as the decision point for which chunking config to keep or test further.